Generate subdivided triangles at various levels of subdivision - needed for rendering a BCS.

In [ ]:
# Run from the repository root, so that `import bcs` and the relative
# data/, configs/ and sdf_weights/ paths work wherever this notebook lives.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bcs').is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import trimesh
import numpy as np

def subdivide(V,F):
    print('refining')
    f_start = F.shape[0]
    v_start = V.shape[0]
    e_start = v_start + f_start - 1 #using Euler's formula V - E + F = 2
    
    f_new = f_start*4 #each face is split into 4 faces
    v_new = v_start + e_start#every old edge contributes 1 new vertex
    
    
    new_F = np.zeros((f_new,3),dtype='int64')
    new_V = np.zeros((v_new ,3))
    #print('new-V shape',new_V.shape)
    
     
    new_V[:v_start,:] = V#copy over the old vertices
    
    midpoint_dict = {}#make dictionary to keep track of which edges have been subdivided and the index of the new vertex
    
    v_count = v_start
    f_count = f_start
    
    for i in range(f_start):
        c0 = F[i,0]
        c1 = F[i,1]
        c2 = F[i,2]
        
        v0 = V[c0,:]
        v1 = V[c1,:]
        v2 = V[c2,:]
        
        if (min(c1,c2),max(c1,c2)) in midpoint_dict:
            
            
            c3 = midpoint_dict[(min(c1,c2),max(c1,c2))]
        else:
            
            c3 = v_count
            
            v3 = (v1+v2)/2.0
            new_V[c3,:] = v3
            midpoint_dict[(min(c1,c2),max(c1,c2))] = c3
            v_count+=1#keep track of. 
            
            
        if (min(c0,c2),max(c0,c2)) in midpoint_dict:
            c4 = midpoint_dict[(min(c0,c2),max(c0,c2))]
            
        else:
            
            c4 = v_count
            
            v4 = (v0+v2)/2.0
            new_V[c4,:] = v4
            midpoint_dict[(min(c0,c2),max(c0,c2))] = c4
            v_count+=1#keep track of. 
        
        
        if (min(c0,c1),max(c0,c1)) in midpoint_dict:
            c5 = midpoint_dict[(min(c0,c1),max(c0,c1))]
            
        else:
            
            c5 = v_count
            
            v5 = (v0+v1)/2.0
            new_V[c5,:] = v5
            midpoint_dict[(min(c0,c1),max(c0,c1))] = c5
            v_count+=1#keep track of. 
  
        
        new_F[i,:] = np.array([c3,c4,c5])
        new_F[f_count,:] = np.array([c3,c2,c4])
        f_count+=1
        new_F[f_count,:] = np.array([c5,c4,c0])
        f_count+=1
        new_F[f_count,:] = np.array([c1,c3,c5])
        f_count+=1
    
    return new_V,new_F





V = np.array([[0.0, 0.0, 0.0],
             [1.0, 0.0, 0.0],
             [0.5, np.sqrt(3)/2.0, 0.0]])

F = np.array([[0,1,2]])

mesh = trimesh.Trimesh(vertices=V, faces=F)
mesh.export('data/high_precision_subdiv_triangles/triangle_0.obj')

for i in range(1,12):
    print('Subdivision',str(i))
    V,F = subdivide(V,F)
    mesh.vertices=V
    mesh.faces=F
    mesh.export('data/high_precision_subdiv_triangles/triangle_'+str(i)+'.obj', digits=10)
    
